## Inicializa o principais dependencias

In [1]:
%run "../0 - Base/inicializa.ipynb"

In [2]:
silverVendas = pd.read_excel("..\\Raw\\table_silver\\silverVendas_ETL.xlsx")

In [3]:
silverVendas['OPERADOR'].value_counts()

OPERADOR
JULIANA           8950
PATRICIA          6012
CARLOS_GER         309
N_IDENTIFICADO     284
DESCONHECIDO       267
Name: count, dtype: int64

In [4]:
silverVendas['COD_PRODUTO'].duplicated().value_counts() # Valida quanto produtos repeditos

COD_PRODUTO
True     15798
False       24
Name: count, dtype: int64

In [5]:
silverVendas['ID_TRANSACAO'].duplicated().value_counts()

ID_TRANSACAO
False    15822
Name: count, dtype: int64

In [6]:
silverVendas['COD_PRODUTO'].isna().value_counts()

COD_PRODUTO
False    15822
Name: count, dtype: int64

In [7]:
silverVendas['COD_PRODUTO'].value_counts()

COD_PRODUTO
1000001    1306
102         684
403         680
302         672
104         667
201         661
105         659
401         650
501         645
999         643
103         632
301         631
303         627
101         623
202         620
203         619
503         614
502         608
304         607
204         606
601         602
602         601
402         600
404         565
Name: count, dtype: int64

-------

------

In [8]:
silverVendas.head(5)

,ID_TRANSACAO,DATA_HORA_VENDA,PDV_CAIXA,OPERADOR,COD_PRODUTO,QTD_VENDIDA,PRECO_UNIT_COBRADO,TOTAL_PAGO,FORMA_PAGTO,STATUS_VENDA,DATA,ANO,MES,DIA,VENDAS_TOTAL,DIF_VENDA
0,900001,2026-01-01,CAIXA_P,JULIANA,503,5,3.94,19.70,FIADO_CADERNETO,CONCLUIDA,2026-01-01,2026,1,1,19.70,0.00
1,900002,2026-01-01,CAIXA_02,PATRICIA,1000001,2,34.47,68.94,FIADO_CADERNETO,CONCLUIDA,2026-01-01,2026,1,1,68.94,0.00
2,900003,2026-01-01,CAIXA_02,PATRICIA,404,4,32.32,129.28,FIADO_CADERNETO,CONCLUIDA,2026-01-01,2026,1,1,129.28,0.00
3,900004,2026-01-01,CAIXA_02,PATRICIA,303,4,36.05,144.20,CARTAO_CRED,CONCLUIDA,2026-01-01,2026,1,1,144.20,0.00
4,900005,2026-01-01,CAIXA_02,PATRICIA,304,5,25.25,126.25,CARTAO_DEB,CONCLUIDA,2026-01-01,2026,1,1,126.25,0.00


# CRIA NOVAS TABELAS

### Status

In [9]:
g_status = silverVendas.groupby('STATUS_VENDA', as_index=False).agg(
      TRANSACOES=('ID_TRANSACAO', 'nunique'),
      QTD_VENDIDA=('QTD_VENDIDA', 'sum'),
      FATURAMENTO_BRUTO=('VENDAS_TOTAL', 'sum'),
)

In [10]:
g_status.head(3)

,STATUS_VENDA,TRANSACOES,QTD_VENDIDA,FATURAMENTO_BRUTO
0,ABERTO,2247,6748,158608.15
1,CANCELADA,2226,6525,0.00
2,CONCLUIDA,9051,27302,645052.34


----

In [11]:
GoldVendasValidas = silverVendas[silverVendas['STATUS_VENDA'] == 'CONCLUIDA'].copy()

g_id = (
    GoldVendasValidas.groupby('ID_TRANSACAO', as_index=False)
    .agg(QTD_VENDIDA=('QTD_VENDIDA', 'sum'), FATURAMENTO=('TOTAL_PAGO', 'sum'))
)

def resumo(coluna):
    return (
        GoldVendasValidas
        .groupby(coluna, as_index=False)
        .agg(
            FATURAMENTO=('TOTAL_PAGO', 'sum'),
            QTD_VENDIDA=('QTD_VENDIDA', 'sum'),
            TRANSACOES=('ID_TRANSACAO', 'nunique'),
        )
        .assign(
            PRECO_MEDIO=lambda d: d['FATURAMENTO'] / d['QTD_VENDIDA'],
            TICKET_MEDIO=lambda d: d['FATURAMENTO'] / d['TRANSACOES'],
        )
        .sort_values('FATURAMENTO', ascending=False)
    )

g_cx = resumo('PDV_CAIXA')
g_op = resumo('OPERADOR')
g_cod = resumo('COD_PRODUTO')
g_fpagamento = resumo('FORMA_PAGTO')
g_ano = resumo('ANO')
g_mes = resumo(['ANO', 'MES'])

In [12]:
silverVendas['COD_PRODUTO'].isna().value_counts()

COD_PRODUTO
False    15822
Name: count, dtype: int64

In [13]:
assert abs(g_mes['FATURAMENTO'].sum() - GoldVendasValidas['TOTAL_PAGO'].sum()) < 0.01
assert abs(g_ano['FATURAMENTO'].sum() - GoldVendasValidas['TOTAL_PAGO'].sum()) < 0.01

----

In [14]:
# cria tabela, agrupa por operador com agregaçãodos id
g_op_x_id = (
    silverVendas
    .groupby('OPERADOR', as_index=False)['ID_TRANSACAO']
    .sum()
)
g_op_x_id.head(10)

,OPERADOR,ID_TRANSACAO
0,CARLOS_GER,280510519
1,DESCONHECIDO,242272139
2,JULIANA,8125314984
3,N_IDENTIFICADO,257931850
4,PATRICIA,5458946261


In [15]:
# cria tabela, agrupa por codigo produto com agregaçãodos id
g_cod_x_id = (
    silverVendas
    .groupby('COD_PRODUTO', as_index=False)['QTD_VENDIDA']
    .sum()
)

g_cod_x_id.head(5)

,COD_PRODUTO,QTD_VENDIDA
0,101,1890
1,102,2067
2,103,1831
3,104,2026
4,105,1953


In [16]:
g_cod_x_id['COD_PRODUTO'].value_counts()

COD_PRODUTO
101        1
102        1
103        1
104        1
105        1
201        1
202        1
203        1
204        1
301        1
302        1
303        1
304        1
401        1
402        1
403        1
404        1
501        1
502        1
503        1
601        1
602        1
999        1
1000001    1
Name: count, dtype: int64

In [17]:
# cria tabela, agrupa por forma com agregaçãodos Qtd vendida
g_form_x_cx = (
    silverVendas
    .groupby('FORMA_PAGTO', as_index=False)['QTD_VENDIDA']
    .count()
)

g_form_x_cx.head(5)

,FORMA_PAGTO,QTD_VENDIDA
0,CARTAO_CRED,1921
1,CARTAO_DEB,2018
2,DINHEIRO,1942
3,FIADO_CADERNETO,2062
4,INDEFINIDO,1996


In [18]:
# cria tabela, agrupa por status com agregaçãodos qt vendas
g_status_x_cod = (
    silverVendas
    .groupby('STATUS_VENDA', as_index=False)['QTD_VENDIDA']
    .count()
)

g_status_x_cod.head(5)

,STATUS_VENDA,QTD_VENDIDA
0,ABERTO,2247
1,CANCELADA,2226
2,CONCLUIDA,9051
3,ESTORNADA,2298


In [19]:
# cria tabela, agrupa por CX com agregaçãodos qtd vendida
g_status_x_unid = (
    silverVendas
    .groupby('PDV_CAIXA', as_index=False)['QTD_VENDIDA']
    .count()
)

g_status_x_unid.head(10)

,PDV_CAIXA,QTD_VENDIDA
0,CAIXA_01,6282
1,CAIXA_02,6371
2,CAIXA_P,3169


In [20]:
# cria tabela, agrupa por operador com agregaçãodos outros dados
g_op_x_agg = (
    silverVendas
    .groupby('OPERADOR')
    .agg(
    TOTAL=('TOTAL_PAGO', 'sum'),
    MEDIA=('TOTAL_PAGO', 'mean'),
    QUANTIDADE=('QTD_VENDIDA', 'count'),
    COD_PRODUTOS=('ID_TRANSACAO', 'nunique')
)
)

g_op_x_agg.head(10)

,TOTAL,MEDIA,QUANTIDADE,COD_PRODUTOS
OPERADOR,,,,
CARLOS_GER,12033.08,38.94,309,309
DESCONHECIDO,10760.96,40.30,267,267
JULIANA,362366.23,40.49,8950,8950
N_IDENTIFICADO,10741.34,37.82,284,284
PATRICIA,249150.73,41.44,6012,6012


----

In [21]:
g_op_x_id.head(3)

,OPERADOR,ID_TRANSACAO
0,CARLOS_GER,280510519
1,DESCONHECIDO,242272139
2,JULIANA,8125314984


In [22]:
g_op.head(3)

,OPERADOR,FATURAMENTO,QTD_VENDIDA,TRANSACOES,PRECO_MEDIO,TICKET_MEDIO
2,JULIANA,362366.23,15365,5094,23.58,71.14
4,PATRICIA,249150.73,10545,3499,23.63,71.21
0,CARLOS_GER,12033.08,497,159,24.21,75.68


-------------

In [23]:
resultado = False

In [24]:
gold_Vendas = silverVendas

def export_dados_excel(resultado):

    pasta = Path("..\\Raw\\table_gold\\")
    pasta.mkdir(parents=True, exist_ok=True)

    if resultado:

        arquivo_excel = pasta / "gold_Vendas.xlsx"

        # 1. Criar os arquivos
        gold_Vendas.to_excel(arquivo_excel, index=False)


        # 3. Consolidar em um Excel com duas abas
        with pd.ExcelWriter(arquivo_excel, engine="openpyxl") as writer:

            gold_Vendas.to_excel(
                writer,
                sheet_name="g_Vendass",
                index=False
            )


            g_cod_x_id.to_excel(
                writer,
                sheet_name="cod_x_id",
                index=False
            )

            g_form_x_cx.to_excel(
                writer,
                sheet_name="form_x_cx",
                index=False
            )

            g_status_x_cod.to_excel(
                writer,
                sheet_name="status_x_cod",
                index=False
            )

            g_status_x_unid.to_excel(
                writer,
                sheet_name="status_x_unid",
                index=False
            )

            g_op_x_agg.to_excel(
                writer,
                sheet_name="op_x_agg",
                index=False
            )

            
            g_id.to_excel(
                writer,
                sheet_name="id",
                index=False
            )

            g_cx.to_excel(
                writer,
                sheet_name="cx",
                index=False
            )

            g_op.to_excel(
                writer,
                sheet_name="op",
                index=False
            )

            g_cod.to_excel(
                writer,
                sheet_name="cod",
                index=False
            )

            g_fpagamento.to_excel(
                writer,
                sheet_name="fpagamento",
                index=False
            )

            g_mes.to_excel(
                writer,
                sheet_name="mes",
                index=False
            )

            g_ano.to_excel(
                writer,
                sheet_name="ano",
                index=False
            )

            
        return True

    print("ERRO")
    return False

In [25]:
export_dados_excel(True)

True